## 1. Environment Setup

Install Required Library

In [ ]:
!pip install pyspark

#### 1.1 Create SparkContext and SparkSession

Remeber to create a session, which is an entry point to Spark

In [1]:
# create entry points to spark
from pyspark.sql import SparkSession

ss  = SparkSession.builder \
                            .master("local[1]")\
                            .appName("SparkByExamples.com")\
                            .getOrCreate()
spark = ss.sparkContext

## 2. TF, IDF and TF-IDF (Important!)
Here, we will use TF-IDF (term frequency–inverse document frequency)
to find the "keyword" mentioned in user text.

* TF is short for **Term Frequency**. It is the number of times that a **term** $t$ appears in document $d$. The higher the TF is for a specific term, the more important that term is to that document.

* IDF is short for **Inverse Document Frequency**. It is the **number of documents** that contains term $t$. If a term exists in every single document, then the Document Frequency is the largest and is 1. And the Inverse Document Frequency will be the smallest. In the situation, this term is non-informative for classifying the documents.The IDF is a measure of the relevance of a term. The higher the IDF is, the more relavant the term is.

* TF-IDF is the product of TF and IDF. A high TF-IDF is obtained when the The Term Frequency is high and the Document Frequency is low (IDF is high).

Formally,
**remark: |D|= total number of documents**

For example, we have two documents and their text as follow:


In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import CountVectorizer, IDF, Tokenizer

# create some toy data
sentenceData = ss.createDataFrame([
    (0, "Python python Spark Spark"),
    (1, "Python SQL")],
    ["document", "sentence"])

sentenceData.show(truncate=False)  # display all result

+--------+-------------------------+
|document|sentence                 |
+--------+-------------------------+
|0       |Python python Spark Spark|
|1       |Python SQL               |
+--------+-------------------------+



Then:
* $ TF(python, doc0) = 2, TF(spark, doc0) = 2, TF(python, doc1) = 1, TF(SQL, doc1) = 1 $  (TF: no. of term in doc)
* $ DF(python, D) = 2, DF(spark, D) = 1, DF(SQL, D) = 1$ (DF: no. of doc containing the term)

* IDF:

$$
\begin{align}
IDF(python, D) = log \frac{|D| + 1}{DF(python,D)+1} = log (\frac{2+1}{2+1}) = 0 \\
IDF(spark, D) = log \frac{|D| + 1}{DF(spark,D)+1} = log (\frac{2+1}{1+1}) = 0.4055 \\
IDF(sql, D) = log \frac{|D| + 1}{DF(sql,D)+1} = log (\frac{2+1}{1+1}) = 0.4055
\end{align}
$$

* TFIDF:

$$
\begin{align}
TFIDF(python, doc0, D) = TF(python, doc0) \cdot IDF(python,D) = 2 * 0 = 0 \\
TFIDF(spark, doc0, D) = TF(spark, doc0) \cdot IDF(spark,D) = 2 * 0.4055 = 0.811 \\
TFIDF(python, doc1, D) = TF(python, doc1) \cdot IDF(python,D) = 1 * 0 = 0 \\
TFIDF(sql, doc1, D) = TF(sql, doc1) \cdot IDF(sql,D) = 1 * 0.4055 = 0.4055 \\
\end{align}
$$



In [ ]:
# define the tokenizer and CountVectorizer
tokenizer = Tokenizer(inputCol="sentence", outputCol="words") # Tokenizer split each document to words by space
tf_  = CountVectorizer(inputCol="words", outputCol="tf") # CountVectorizer count word freq of each doc (i.e., TF)
idf_ = IDF(inputCol="tf", outputCol="tfidf")

# step-by-step
wordsData = tokenizer.transform(sentenceData)

## fit and transform model: build TF model and apply on data
tf_model = tf_.fit(wordsData) # create TF model on wordsData
tf = tf_model.transform(wordsData) # apply on the data
tf.show(4)

## fit and transform model: build IDF model and apply on data
idf_model = idf_.fit(tf)
tfidf = idf_model.transform(tf) # By default, Pyspark computes the IDF vector, scale the term frequencies by IDF and directly output TFIDF.
tfidf.show(4, truncate=False)


+--------+--------------------+--------------------+-------------------+
|document|            sentence|               words|                 tf|
+--------+--------------------+--------------------+-------------------+
|       0|Python python Spa...|[python, python, ...|(3,[0,1],[2.0,2.0])|
|       1|          Python SQL|       [python, sql]|(3,[0,2],[1.0,1.0])|
+--------+--------------------+--------------------+-------------------+

+--------+-------------------------+------------------------------+-------------------+----------------------------------+
|document|sentence                 |words                         |tf                 |tfidf                             |
+--------+-------------------------+------------------------------+-------------------+----------------------------------+
|0       |Python python Spark Spark|[python, python, spark, spark]|(3,[0,1],[2.0,2.0])|(3,[0,1],[0.0,0.8109302162163288])|
|1       |Python SQL               |[python, sql]                 |(3

In Spark, features created by transform will be in a format of (N, \[I], \[F])
where **N**: no. of features (i.e., word), **I**: their word index and **F**: their count (e.g., TF, IDF)

In [ ]:
from pyspark.sql.types import ArrayType, StringType

def termsIdx2Term(vocabulary):
    def termsIdx2Term(termIndices):
        return [vocabulary[int(index)] for index in termIndices]
    return udf(termsIdx2Term, ArrayType(StringType()))

In [ ]:
# Showcase how to break down of tfidf display format
from pyspark.sql.functions import udf
import pyspark.sql.functions as F
from pyspark.sql.types import  StringType, DoubleType, IntegerType, ArrayType

indices_udf = udf(lambda vector: vector.indices.tolist(), ArrayType(IntegerType()))
values_udf = udf(lambda vector: vector.toArray().tolist(), ArrayType(DoubleType()))

rawFeatures = idf_model.transform(tf).select('tfidf')
rawFeatures.withColumn('indices', indices_udf(F.col('tfidf')))\
           .withColumn('values', values_udf(F.col('tfidf')))\
           .withColumn("Terms", termsIdx2Term(tf_model.vocabulary)("indices")).show(truncate=False)

+----------------------------------+-------+------------------------------+---------------+
|tfidf                             |indices|values                        |Terms          |
+----------------------------------+-------+------------------------------+---------------+
|(3,[0,1],[0.0,0.8109302162163288])|[0, 1] |[0.0, 0.8109302162163288, 0.0]|[python, spark]|
|(3,[0,2],[0.0,0.4054651081081644])|[0, 2] |[0.0, 0.0, 0.4054651081081644]|[python, sql]  |
+----------------------------------+-------+------------------------------+---------------+



### 2.1 CountVectorizer

In previous example, we also try **`CountVectorizer()`** function.
The **`CountVectorizer()`** function has three parameters to control which terms will be kept as features.

* minTF: features that has term frequency less than minTF will be removed. If minTF=1, then no features will be removed.
* minDF: features that has document frequency less than minDF will be removed. If minDF=1, then no features will be removed.
* vocabSize: keep terms of the top vocabSize frequencies.

In the example below, the `minTF=1.0,minDF=1.0` and `vocabSize=20`, which is larger than the total number of terms. Therefore, all features (terms) will be kept.

In [ ]:
from pyspark.ml.feature import CountVectorizer, Tokenizer
from pyspark.ml import Pipeline

# create some toy data
sentenceData = ss.createDataFrame([
    (0, "Python python Spark Spark"),
    (1, "Python SQL")],
    ["document", "sentence"])

# create count vectorizer
tokenizer = Tokenizer(inputCol="sentence", outputCol="words") # Tokenizer split each document to words by space
countvectorizer  = CountVectorizer(minTF=1.0, minDF=1.0, vocabSize=20, \
inputCol="words", outputCol='features(vocabSize), [word index], [term frequency]') # CountVectorizer count word freq of each doc

# one-step pipeline
pipeline = Pipeline(stages=[tokenizer, countvectorizer])

# fit and transform model
pipeline.fit(sentenceData).transform(sentenceData).show(truncate=False)

+--------+-------------------------+------------------------------+---------------------------------------------------+
|document|sentence                 |words                         |features(vocabSize), [word index], [term frequency]|
+--------+-------------------------+------------------------------+---------------------------------------------------+
|0       |Python python Spark Spark|[python, python, spark, spark]|(3,[0,1],[2.0,2.0])                                |
|1       |Python SQL               |[python, sql]                 |(3,[0,2],[1.0,1.0])                                |
+--------+-------------------------+------------------------------+---------------------------------------------------+



## 2.2 Text Data Cleaning
TFIDF is a technique to quantify the **word features** (e.g., the word importance) in a set of documents.
In real-world data, however, we are interested in not only word, but also **multi-word phrase** features.
For example, the phrase *'don't like'* can imply that the customer hates the product, if we consider only a single word (i.e., like), we may have wrong analytics.
Additionally, some frequent words (e.g., Hello) are less informative and many time irrelevant in our analysis.

**Thus, taking careful consideration into data cleaning** can often improve model performance in big data analytics.

### 2.2.1 NGram
Here, we use ``n-gram()`` to split each text into **bi-gram** (i.e., a two-words phrases).

In [ ]:
from pyspark.ml.feature import NGram

wordDataFrame = ss.createDataFrame([
        (0, ["Hi", "I", "heard", "about", "Spark"]),
        (1, ["I", "wish", "Java", "could", "use", "case", "classes"]),
        (2, ["Logistic", "regression", "models", "are", "neat"])
        ], ["id", "words"])

ngram = NGram(n=2, inputCol="words", outputCol="ngrams")

# ngramDataFrame = ngram.transform(wordDataFrame)
# ngramDataFrame.select("ngrams").show(truncate=False)

countvectorizer  = CountVectorizer(minTF=1.0, minDF=1.0, vocabSize=20, \
inputCol="ngrams", outputCol='features(ngrams)') # CountVectorizer count word freq of each doc

# one-step pipeline
pipeline = Pipeline(stages=[ngram, countvectorizer])

# fit and transform model
result = pipeline.fit(wordDataFrame).transform(wordDataFrame)
result.select('ngrams', 'features(ngrams)').show(truncate=False) # format: 'features(vocabSize), [word index], [term frequency]'


+------------------------------------------------------------------+----------------------------------------------+
|ngrams                                                            |features(ngrams)                              |
+------------------------------------------------------------------+----------------------------------------------+
|[Hi I, I heard, heard about, about Spark]                         |(14,[1,2,10,13],[1.0,1.0,1.0,1.0])            |
|[I wish, wish Java, Java could, could use, use case, case classes]|(14,[4,5,6,8,11,12],[1.0,1.0,1.0,1.0,1.0,1.0])|
|[Logistic regression, regression models, models are, are neat]    |(14,[0,3,7,9],[1.0,1.0,1.0,1.0])              |
+------------------------------------------------------------------+----------------------------------------------+



### 2.2.2 ``StopWordsRemover()``
Some words are frequent words that are not informative.
For example, I, a, an, the.
We use ``StopWordsRemover()`` to take away these words.

In [ ]:
from pyspark.ml.feature import StopWordsRemover

sentenceData = ss.createDataFrame([
    (0, ["I", "saw", "the", "red", "balloon"]),
    (1, ["Mary", "had", "a", "little", "lamb"])
], ["id", "raw"])

remover = StopWordsRemover(inputCol="raw", outputCol="removeded")
remover.transform(sentenceData).show(truncate=False)

+---+----------------------------+--------------------+
|id |raw                         |removeded           |
+---+----------------------------+--------------------+
|0  |[I, saw, the, red, balloon] |[saw, red, balloon] |
|1  |[Mary, had, a, little, lamb]|[Mary, little, lamb]|
+---+----------------------------+--------------------+



## 2.3 Text Information Extraction
Here, we will introduce few methods to extract useful text feature from text.
They are:
* **POS tagging**: the process of getting the word form of text (e.g., noun, verb, adjective)
* **Chunking**: the process of grouping noun phrase (e.g., Brat Pit as Brad_Pitt)
* **Normalization**: the process of transforming text into a single canonical form, e.g., converting text to lowercase, removing punctuations and so on.
* **Stemming**: the process of reducing inflected words to their **word stem**.
* **Lemmatization**: the process of grouping variant forms of the same word so that they can be analyzed as a single item.



First, we install nltk, download nltk package and create a toy example

In [ ]:
!pip install nltk

import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [ ]:
from pyspark.sql.types import StringType, IntegerType

# create a example text dataframe
text_list = ['Peter Parker is a nice guy and lives in New York.', 'Bruce Wayne is also a nice guy and lives in Gotham City.']

textDF = ss.createDataFrame(text_list, StringType()).toDF("text")
textDF.show(truncate=80)

+--------------------------------------------------------+
|                                                    text|
+--------------------------------------------------------+
|       Peter Parker is a nice guy and lives in New York.|
|Bruce Wayne is also a nice guy and lives in Gotham City.|
+--------------------------------------------------------+



### 2.3.1 Part-of-Speech (POS) Tagging
This can identify the POS of word (e.g., noun, verb, adjective)

POS name ref: https://bw-desc.de/index.php/wiki/penn-treebank-tagset/

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import *
import nltk

## define udf function
def sent_to_tag_words(sent):
    wordlist = nltk.word_tokenize(sent)
    tagged_words = nltk.pos_tag(wordlist)
    return(tagged_words)
## define schema for returned result from the udf function
## the returned result is a list of tuples.
schema = ArrayType(StructType([
            StructField('f1', StringType()),
            StructField('f2', StringType())
        ]))

## the udf function
sent_to_tag_words_udf = udf(sent_to_tag_words, schema)

# apply on text
df_tagged_words = textDF.select('text', sent_to_tag_words_udf(textDF.text).alias('tagged_words'))
df_tagged_words.show(5, truncate=False)

+--------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------+
|text                                                    |tagged_words                                                                                                                                              |
+--------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------+
|Peter Parker is a nice guy and lives in New York.       |[{Peter, NNP}, {Parker, NNP}, {is, VBZ}, {a, DT}, {nice, JJ}, {guy, NN}, {and, CC}, {lives, NNS}, {in, IN}, {New, NNP}, {York, NNP}, {., .}]              |
|Bruce Wayne is also a nice guy and lives in Gotham City.|[{Bruce, NNP}, {Wayne, NNP}, {is, VBZ}, {also, RB}, {a, DT}, {nice, JJ}, {guy, NN}, {a

### 2.3.2 Chunking

Chunking is the process of segmenting and labeling multitokens. The following example shows how to do a noun phrase chunking on the tagged words data frame from the previous step.

First we define a *udf* function which chunks noun phrases from a list of pos-tagged words.

In [ ]:
import nltk
from pyspark.sql.functions import udf
from pyspark.sql.types import *

# define a udf function to chunk noun phrases from pos-tagged words
grammar = "NP: {<DT>?<JJ>*<NN>}"
chunk_parser = nltk.RegexpParser(grammar)

## the udf function
chunk_parser_udf = udf(lambda x: str(chunk_parser.parse(x)), StringType())

df_NP_chunks = df_tagged_words.select(chunk_parser_udf(df_tagged_words.tagged_words).alias('NP_chunk'))
df_NP_chunks.show(5, truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------------------+
|NP_chunk                                                                                                                                       |
+-----------------------------------------------------------------------------------------------------------------------------------------------+
|(S\n  Peter/NNP\n  Parker/NNP\n  is/VBZ\n  (NP a/DT nice/JJ guy/NN)\n  and/CC\n  lives/NNS\n  in/IN\n  New/NNP\n  York/NNP\n  ./.)             |
|(S\n  Bruce/NNP\n  Wayne/NNP\n  is/VBZ\n  also/RB\n  (NP a/DT nice/JJ guy/NN)\n  and/CC\n  lives/VBZ\n  in/IN\n  Gotham/NNP\n  City/NNP\n  ./.)|
+-----------------------------------------------------------------------------------------------------------------------------------------------+



### 2.3.3 Stemming
Stemming is the process of reducing inflected (or sometimes derived) words to their word stem (e.g., *changing*, *changes*, *change* > *chang*).

In [ ]:
import nltk
from nltk.stem.wordnet import WordNetLemmatizer
from nltk.stem import PorterStemmer
from pyspark.sql.types import StringType, IntegerType

# this package is needed to perform stemming
ps = PorterStemmer()

def stemming(data_str):
    # expects a string
    cleaned_str = ''
    text = data_str.split()

    for word in text:
        cleaned_str+= ps.stem(word) + ' '

    return cleaned_str

## the udf function
sent_to_stem_words_udf = udf(lambda x: str(stemming(x)), StringType())

# create a example text dataframe
text_list = ['Peter Parker is a nice guy and lives in New York.', 'Bruce Wayne is also a nice guy and lives in Gotham City.']

textDF = ss.createDataFrame(text_list, StringType()).toDF("text")

df_stem_words = textDF.select('text', sent_to_stem_words_udf(textDF.text).alias('stemmed_text'))
df_stem_words.show(5, truncate=False)


+--------------------------------------------------------+-------------------------------------------------------+
|text                                                    |stemmed_text                                           |
+--------------------------------------------------------+-------------------------------------------------------+
|Peter Parker is a nice guy and lives in New York.       |peter parker is a nice guy and live in new york.       |
|Bruce Wayne is also a nice guy and lives in Gotham City.|bruce wayn is also a nice guy and live in gotham city. |
+--------------------------------------------------------+-------------------------------------------------------+



### 2.3.4 Lemmatization
In Lemmatization root word is called Lemma.
A lemma (plural lemmas or lemmata) is the canonical form, dictionary form, or citation form of a set of words.
For example, *runs*, *running*, *ran* are all forms of the word *run*, therefore *run* is the lemma of all these words


In [ ]:
import nltk
from nltk.stem.wordnet import WordNetLemmatizer

def lemmatize(data_str):
    # expects a string
    list_pos = 0
    cleaned_str = ''
    lmtzr = WordNetLemmatizer()
    text = data_str.split()
    tagged_words = nltk.pos_tag(text)
    for word in tagged_words:
        if 'v' in word[1].lower():
            lemma = lmtzr.lemmatize(word[0], pos='v')
        else:
            lemma = lmtzr.lemmatize(word[0], pos='n')
        if list_pos == 0:
            cleaned_str = lemma
        else:
            cleaned_str = cleaned_str + ' ' + lemma
        list_pos += 1
    return cleaned_str

## the udf function
sent_to_lem_words_udf = udf(lambda x: str(lemmatize(x)), StringType())

df_tagged_words = textDF.select('text', sent_to_lem_words_udf(textDF.text).alias('lemmatized_text'))
df_tagged_words.show(5, truncate=False)


+--------------------------------------------------------+-------------------------------------------------------+
|text                                                    |lemmatized_text                                        |
+--------------------------------------------------------+-------------------------------------------------------+
|Peter Parker is a nice guy and lives in New York.       |Peter Parker be a nice guy and life in New York.       |
|Bruce Wayne is also a nice guy and lives in Gotham City.|Bruce Wayne be also a nice guy and live in Gotham City.|
+--------------------------------------------------------+-------------------------------------------------------+



### 2.3.5 Normalization
The process of transforming text into a single canonical form, e.g., converting text to lowercase, removing punctuations and so on.

In [ ]:
import re, string

def normalize(data_str):
    # compile regex
    url_re = re.compile('https?://(www.)?\w+\.\w+(/\w+)*/?')
    punc_re = re.compile('[%s]' % re.escape(string.punctuation))
    num_re = re.compile('(\\d+)')
    mention_re = re.compile('@(\w+)')
    alpha_num_re = re.compile("^[a-z0-9_.]+$")
    # convert to lowercase
    data_str = data_str.lower()
    # remove hyperlinks
    data_str = url_re.sub(' ', data_str)
    # remove @mentions
    data_str = mention_re.sub(' ', data_str)
    # remove puncuation
    data_str = punc_re.sub(' ', data_str)
    # remove numeric 'words'
    data_str = num_re.sub(' ', data_str)
    # remove non a-z 0-9 characters and words shorter than 3 characters
    list_pos = 0
    cleaned_str = ''
    for word in data_str.split():
        if list_pos == 0:
            if alpha_num_re.match(word) and len(word) > 2:
                cleaned_str = word
            else:
                cleaned_str = ' '
        else:
            if alpha_num_re.match(word) and len(word) > 2:
                cleaned_str = cleaned_str + ' ' + word
            else:
                cleaned_str += ' '
        list_pos += 1
    return cleaned_str


## the udf function
sent_to_normalize_words_udf = udf(lambda x: str(normalize(x)), StringType())

# apply on text
df_normed_words = textDF.select('text', sent_to_normalize_words_udf(textDF.text).alias('normalized_words'))
df_normed_words.show(5, truncate=False)

<>:5: SyntaxWarning: invalid escape sequence '\w'
<>:8: SyntaxWarning: invalid escape sequence '\w'
<>:5: SyntaxWarning: invalid escape sequence '\w'
<>:8: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipython-input-1315478425.py:5: SyntaxWarning: invalid escape sequence '\w'
  url_re = re.compile('https?://(www.)?\w+\.\w+(/\w+)*/?')
/tmp/ipython-input-1315478425.py:8: SyntaxWarning: invalid escape sequence '\w'
  mention_re = re.compile('@(\w+)')


+--------------------------------------------------------+--------------------------------------------------+
|text                                                    |normalized_words                                  |
+--------------------------------------------------------+--------------------------------------------------+
|Peter Parker is a nice guy and lives in New York.       |peter parker   nice guy and lives  new york       |
|Bruce Wayne is also a nice guy and lives in Gotham City.|bruce wayne  also  nice guy and lives  gotham city|
+--------------------------------------------------------+--------------------------------------------------+



## 3. TODO Exercises

<font color='#0000FF'> TODO 0: Write a module to read setenceData as RDD, then split the text by space and lowercase the word. Finally, do the following counts:
* **No. of Document**: Number of document
* **Term_Frequency**: Words count of each document in any output format. E.g., ((Doc ID, word), count)((0, 'studying'), 2), (1, 'studying;), 1)
* **Document_Frequency**: No. of document containing a word. E.g., (Word, Count, ('studying', 2) (15 marks)</font>

<font color='#0000FF'> **Hints**: You may need to use the following functions (also look into Example 3.4.5):

* `lower()`
* `split()`
* `flatMapValues()`
* `distinct()`
* `count()`
* `countByKey()`
* `countByValue()`
</font>

In [2]:
#TODO 0. Write a module to read setenceData as RDD, then split the text by space and lowercase the word.
#        Finally, do the following counts:
#        No. of Document: Number of document
#        Term_Frequency: Words count of each document in any output format. E.g., ((Doc ID, word), count)((0, 'studying'), 2), (1, 'studying;), 1)
#.       Document_Frequency: No. of document containing a word. E.g., (Word, Count, ('studying', 2)
sentenceData = ss.createDataFrame([
    (0, "I am Billy studying in Computer and I am also studying in Business Course"),
    (1, "Ada is studying in Computer"),
    (2, "Billy and Ada know each other in a computer course course")],
    ["document", "sentence"])

def text_processing(text):
  return [w.lower() for w in text.split()]

## TODO: print out the number of documents.
tokenized_text = sentenceData.rdd.map(lambda x: (x.document, text_processing(x.sentence)))
num_of_documents = len(tokenized_text.collect())
print('Number of documents:', num_of_documents)

## TODO: count words in each document
wordItems = tokenized_text.flatMapValues(lambda x: x).countByValue().items()

print("\nTerm_Frequency:")
for word, cnt in wordItems:
  print(word[1]+":", cnt, "in D" + str(word[0]))

print("\nDocument Frequency:")
## TODO: No of documents containing a term
df = tokenized_text.flatMapValues(lambda x: x).distinct().map(lambda x:(x[1], x[0])).countByKey()
df.items()
for word, cnt in df.items():
  print(word, ":", cnt, sep='', end='|')

Number of documents: 3

Term_Frequency:
i: 2 in D0
am: 2 in D0
billy: 1 in D0
studying: 2 in D0
in: 2 in D0
computer: 1 in D0
and: 1 in D0
also: 1 in D0
business: 1 in D0
course: 1 in D0
ada: 1 in D1
is: 1 in D1
studying: 1 in D1
in: 1 in D1
computer: 1 in D1
billy: 1 in D2
and: 1 in D2
ada: 1 in D2
know: 1 in D2
each: 1 in D2
other: 1 in D2
in: 1 in D2
a: 1 in D2
computer: 1 in D2
course: 2 in D2

Document Frequency:
i:1|am:1|billy:2|studying:2|in:3|computer:3|and:2|also:1|business:1|course:2|ada:2|is:1|know:1|each:1|other:1|a:1|

In [4]:
# TODO 1.1
# In TODO1, you computed the required RDDs for TF-IDF calculation. Now implement TF-IDF scoring
# to identify important terms in each document using only RDD operations.
# You must manually compute TF-IDF (no DataFrames/libraries allowed)
# Only pure RDD operations allowed include: mapValues, map, flatMapValues, join, and sortBy.
# Complete the following code to manually compute TF-IDF scores using raw term frequency and
# the smoothed IDF formula log((n+1)/(df+1)):

from operator import add
import math

...

[('also', 0.6931471805599453, 0),
 ('am', 1.3862943611198906, 0),
 ('and', 0.28768207245178085, 0),
 ('billy', 0.28768207245178085, 0),
 ('business', 0.6931471805599453, 0),
 ('computer', 0.0, 0),
 ('course', 0.28768207245178085, 0),
 ('i', 1.3862943611198906, 0),
 ('in', 0.0, 0),
 ('studying', 0.5753641449035617, 0),
 ('ada', 0.28768207245178085, 1),
 ('computer', 0.0, 1),
 ('in', 0.0, 1),
 ('is', 0.6931471805599453, 1),
 ('studying', 0.28768207245178085, 1),
 ('a', 0.6931471805599453, 2),
 ('ada', 0.28768207245178085, 2),
 ('and', 0.28768207245178085, 2),
 ('billy', 0.28768207245178085, 2),
 ('computer', 0.0, 2),
 ('course', 0.5753641449035617, 2),
 ('each', 0.6931471805599453, 2),
 ('in', 0.0, 2),
 ('know', 0.6931471805599453, 2),
 ('other', 0.6931471805599453, 2)]

In [5]:
# TODO 1.2
# Repeat the TF-IDF computation from TODO3, but now you are allowed to use Spark DataFrame features
# to simplify the logic required to compute the TF-IDF scores.
# You must still manually compute TF-IDF
# (no external libraries allowed) using the RDDs from TODO2.

from operator import add
import math
from pyspark.sql.functions import col

...

+--------+-------------------+------+
|word    |tfidf              |doc_id|
+--------+-------------------+------+
|also    |0.6931471805599453 |0     |
|am      |1.3862943611198906 |0     |
|and     |0.28768207245178085|0     |
|billy   |0.28768207245178085|0     |
|business|0.6931471805599453 |0     |
|computer|0.0                |0     |
|course  |0.28768207245178085|0     |
|i       |1.3862943611198906 |0     |
|in      |0.0                |0     |
|studying|0.5753641449035617 |0     |
|ada     |0.28768207245178085|1     |
|computer|0.0                |1     |
|in      |0.0                |1     |
|is      |0.6931471805599453 |1     |
|studying|0.28768207245178085|1     |
|a       |0.6931471805599453 |2     |
|ada     |0.28768207245178085|2     |
|and     |0.28768207245178085|2     |
|billy   |0.28768207245178085|2     |
|computer|0.0                |2     |
|course  |0.5753641449035617 |2     |
|each    |0.6931471805599453 |2     |
|in      |0.0                |2     |
|know    |0.

<font color='blue'>TODO 2: Create a function to filter out words that are not noun, verb or adjective and apply it on the following text:</font>


<font color='blue'>**Peter Parker is a nice guy and lives in New York. Bruce Wayne is also a nice guy and lives in Gotham City.**</font>

In [ ]:
text_list = ['Peter Parker is a nice guy and lives in New York.', 'Bruce Wayne is also a nice guy and lives in Gotham City.']
textDF = ss.createDataFrame(text_list, StringType()).toDF('text')

def tag_and_remove(data_str):
  ...

send_to_tag_words = udf(...)


df_tagged_words = ...
df_tagged_words.show(2, truncate=False)

+--------------------------------------------------------+-------------------------------------------+
|text                                                    |filtered_text                              |
+--------------------------------------------------------+-------------------------------------------+
|Peter Parker is a nice guy and lives in New York.       |Peter Parker is nice guy lives New York.   |
|Bruce Wayne is also a nice guy and lives in Gotham City.|Bruce Wayne is nice guy lives Gotham City. |
+--------------------------------------------------------+-------------------------------------------+

